---

**Navigation** : [<< Planners-1 Introduction C#](Planners-1-Introduction-Csharp.ipynb) | [Index](../../README.md) | [Planners-2 PDDL Basics >>](Planners-2-PDDL-Basics.ipynb)

---

# Planners-3 : Recherche dans l'Espace d'Etats — twin C# (BFS / DFS / Greedy / A*)

> **Twin C#** de [Planners-3-State-Space](Planners-3-State-Space.ipynb) (Python + networkx). **Tranche 2** du marathon Planners (#4956) : apres Planners-1 (STRIPS + BFS), la progression **search nu -> A*** avec heuristiques, admissibilite et coherence.

## Complementarite (#3801 Prong B)

| Twin | Outil | Valeur |
|------|-------|--------|
| Python (networkx + matplotlib) | visualisation graphe + heatmap terrain | explorer les paysages de recherche visuellement |
| **This (.NET BCL seule)** | `PriorityQueue` .NET 9 + from-scratch | **comprendre** la mecanique interne des algos (frontiere, f=g+h, closed set) |

Pas d'equivalent NuGet didactique idiomatique pour ce niveau -> from-scratch = la valeur. BCL .NET seule, 0 NuGet.

**Le point cle (#3801 Prong B, cas non-degenere)** : sur un terrain uniforme BFS = A*. Sur un **terrain pondere**, A* trouve un chemin **plus long en etapes mais moins couteux** que BFS — c'est la ou A* discrimine vraiment. Le twin Python le demontre visuellement (heatmap) ; le twin C# le demontre **numeriquement** (table Pas/Cout/Nœuds).

## Objectifs d'apprentissage

A la fin de ce notebook, vous saurez :
1. **Representer** un probleme comme un graphe d'etats (etats + successeurs + couts).
2. **Implementer** BFS, DFS, Greedy Best-First et A* from-scratch (.NET 9 `PriorityQueue`).
3. **Distinguer** optimalite-en-pas (BFS) vs optimalite-en-cout (A*).
4. **Justifier** l'admissibilite d'une heuristique (Manhattan sur grille).

## 1. Representation d'un espace d'etats

Un probleme de recherche = (etat initial, test-but, fonction successeurs). Ici une **grille 2D** : un etat = une position `(x, y)` ; les successeurs = les 4 voisins (deplacements N/S/E/O). Le cout d'un pas depend du **terrain** (cas le plus general).


In [1]:
// Modele : etat = position (x, y) sur une grille. Le cout d'un pas depend du terrain d'arrivee.
using System;
using System.Collections.Generic;
using System.Linq;

// Record : egalite structurelle (deux GridState avec memes x,y sont egaux) = necessaire pour HashSet/Dict.
public record GridState(int X, int Y)
{
    public override string ToString() => $"({X},{Y})";
}

// Directions cardinales : Nord/Sud/Est/Ouest.
static readonly (int dx, int dy)[] ACTIONS = { (0, 1), (0, -1), (1, 0), (-1, 0) };

// Terrain pondere : cout d'entree dans une case. Defaut = cout uniforme 1 (cas degenere BFS=A*).
static int CoutCase(GridState s, HashSet<GridState> marais, int coutMarais) =>
    marais.Contains(s) ? coutMarais : 1;

Console.WriteLine($"Modele grille : GridState = record (x,y) ; ACTIONS = {ACTIONS.Length} cardinales.");
Console.WriteLine("Terrain pondere : cout marais > cout normal (sinon, cas degenere ou BFS=A*).");


The below script needs to be able to find the current output cell; this is an easy method to get it.

Modele grille : GridState = record (x,y) ; ACTIONS = 4 cardinales.


Terrain pondere : cout marais > cout normal (sinon, cas degenere ou BFS=A*).


## 2. BFS — recherche en largeur (optimal en nombre de pas)

BFS explore en largeur (file FIFO). Sur un graphe a **cout uniforme**, BFS garantit le chemin le plus court en **nombre d'etapes**. Il explore beaucoup de nœuds (pas d'heuristique pour guider).

In [2]:
// BFS generique : file FIFO, closed set pour eviter les revisites. Garantit le chemin en PAS minimum.
// getSuccessors renvoie (voisin, cout) — le cout est ignore par BFS (qui compte les pas).
static (List<GridState> path, int stepsCost, int nodesExpanded) BFS(
    GridState initial, GridState goal, Func<GridState, IEnumerable<(GridState n, int cost)>> getSuccessors)
{
    var frontier = new Queue<GridState>();
    frontier.Enqueue(initial);
    var cameFrom = new Dictionary<GridState, GridState?> { [initial] = null };
    int nodes = 0;
    while (frontier.Count > 0)
    {
        var cur = frontier.Dequeue();
        nodes++;
        if (cur == goal) return (Reconstruct(cameFrom, goal), stepsCost: 0, nodesExpanded: nodes);
        foreach (var (n, cost) in getSuccessors(cur))
            if (!cameFrom.ContainsKey(n))
            {
                cameFrom[n] = cur;
                frontier.Enqueue(n);
            }
    }
    return (new(), 0, nodes);  // pas de chemin
}

static List<GridState> Reconstruct(Dictionary<GridState, GridState?> cameFrom, GridState goal)
{
    var path = new List<GridState>();
    GridState? cur = goal;
    while (cur is not null)
    {
        path.Add(cur);
        cur = cameFrom[cur];   // cameFrom[x] = parent (ou null si x = initial)
    }
    path.Reverse();
    return path;
}

Console.WriteLine("BFS : file FIFO + closed set. Optimal en NOMBRE DE PAS (cout uniforme).");


BFS : file FIFO + closed set. Optimal en NOMBRE DE PAS (cout uniforme).



(25,67): warning CS8632: L'annotation pour les types référence Nullable doit être utilisée uniquement dans le code au sein d'un contexte d'annotations '#nullable'.

(8,55): warning CS8632: L'annotation pour les types référence Nullable doit être utilisée uniquement dans le code au sein d'un contexte d'annotations '#nullable'.

(28,14): warning CS8632: L'annotation pour les types référence Nullable doit être utilisée uniquement dans le code au sein d'un contexte d'annotations '#nullable'.



## 3. DFS — recherche en profondeur (ni optimal ni complet sans limite)

DFS plonge au maximum avant de revenir en arriere (pile LIFO). Ni optimal ni complet sans limite de profondeur. Interet pedagogique : contraster avec BFS.

In [3]:
// DFS generique : pile LIFO + closed set. Ni optimal ni complet sans limite, mais contraste avec BFS.
static (List<GridState> path, int nodesExpanded) DFS(
    GridState initial, GridState goal, Func<GridState, IEnumerable<(GridState n, int cost)>> getSuccessors,
    int maxDepth = 1000)
{
    var stack = new Stack<(GridState s, int depth)>();
    stack.Push((initial, 0));
    var cameFrom = new Dictionary<GridState, GridState?> { [initial] = null };
    int nodes = 0;
    while (stack.Count > 0)
    {
        var (cur, depth) = stack.Pop();
        nodes++;
        if (cur == goal) return (Reconstruct(cameFrom, goal), nodes);
        if (depth >= maxDepth) continue;
        foreach (var (n, cost) in getSuccessors(cur))
            if (!cameFrom.ContainsKey(n))
            {
                cameFrom[n] = cur;
                stack.Push((n, depth + 1));
            }
    }
    return (new(), nodes);
}

Console.WriteLine("DFS : pile LIFO. Ni optimal ni complet, contraste pedagogique avec BFS.");


DFS : pile LIFO. Ni optimal ni complet, contraste pedagogique avec BFS.



(8,55): warning CS8632: L'annotation pour les types référence Nullable doit être utilisée uniquement dans le code au sein d'un contexte d'annotations '#nullable'.



## 4. Heuristique + Greedy Best-First

Une **heuristique** `h(n)` estime le cout restant d'un nœud au but. Sur une grille, la distance **Manhattan** `|dx|+|dy|` est admissible (ne surestime jamais le vrai cout sur grille sans obstacles diagonaux). Greedy Best-First n'utilise que `h(n)` pour guider — rapide mais **non optimal** (peut se laisser pieger par l'heuristique).

In [4]:
// Heuristique de Manhattan : |dx| + |dy|. Admissible sur grille (ne surestime jamais).
static int Manhattan(GridState a, GridState b) => Math.Abs(a.X - b.X) + Math.Abs(a.Y - b.Y);

// Greedy Best-First : priorite = h(n) seulement. Rapide MAIS non optimal (se laisse pieger par h).
static (List<GridState> path, int stepsCost, int nodesExpanded) GreedyBestFirst(
    GridState initial, GridState goal, Func<GridState, IEnumerable<(GridState n, int cost)>> getSuccessors,
    Func<GridState, GridState, int> heuristic)
{
    var frontier = new PriorityQueue<GridState, int>();
    frontier.Enqueue(initial, heuristic(initial, goal));
    var cameFrom = new Dictionary<GridState, GridState?> { [initial] = null };
    int nodes = 0;
    while (frontier.Count > 0)
    {
        var cur = frontier.Dequeue();
        nodes++;
        if (cur == goal) return (Reconstruct(cameFrom, goal), stepsCost: 0, nodesExpanded: nodes);
        foreach (var (n, cost) in getSuccessors(cur))
            if (!cameFrom.ContainsKey(n))
            {
                cameFrom[n] = cur;
                frontier.Enqueue(n, heuristic(n, goal));
            }
    }
    return (new(), 0, nodes);
}

Console.WriteLine("Manhattan = |dx|+|dy| (admissible sur grille). Greedy : priorite = h(n) seul (non optimal).");


Manhattan = |dx|+|dy| (admissible sur grille). Greedy : priorite = h(n) seul (non optimal).



(11,55): warning CS8632: L'annotation pour les types référence Nullable doit être utilisée uniquement dans le code au sein d'un contexte d'annotations '#nullable'.



## 5. A* — cout cumule g + heuristique h (optimal si h admissible)

A* combine le **cout deja parcouru** `g(n)` et l'**estimation restante** `h(n)` : `f(n) = g(n) + h(n)`. Avec une heuristique **admissible** (h ne surestime jamais), A* est **optimal en cout**. C'est l'algorithme de reference.

In [5]:
// A* : f = g + h. Suit g (cout cumule reel). Optimal en COUT si h admissible. .NET 9 PriorityQueue.
static (List<GridState> path, int cost, int nodesExpanded) AStar(
    GridState initial, GridState goal, Func<GridState, IEnumerable<(GridState n, int cost)>> getSuccessors,
    Func<GridState, GridState, int> heuristic)
{
    var frontier = new PriorityQueue<GridState, int>();
    frontier.Enqueue(initial, heuristic(initial, goal));
    var gScore = new Dictionary<GridState, int> { [initial] = 0 };
    var cameFrom = new Dictionary<GridState, GridState?> { [initial] = null };
    int nodes = 0;
    while (frontier.Count > 0)
    {
        var cur = frontier.Dequeue();
        nodes++;
        if (cur == goal) return (Reconstruct(cameFrom, goal), gScore[goal], nodesExpanded: nodes);
        foreach (var (n, stepCost) in getSuccessors(cur))
        {
            int tentative = gScore[cur] + stepCost;
            if (tentative < gScore.GetValueOrDefault(n, int.MaxValue))
            {
                cameFrom[n] = cur;
                gScore[n] = tentative;
                frontier.Enqueue(n, tentative + heuristic(n, goal));  // f = g + h
            }
        }
    }
    return (new(), 0, nodes);
}

Console.WriteLine("A* : f = g + h. Optimal en COUT si h admissible. PriorityQueue<GridState,int> (.NET 9).");


A* : f = g + h. Optimal en COUT si h admissible. PriorityQueue<GridState,int> (.NET 9).



(9,55): warning CS8632: L'annotation pour les types référence Nullable doit être utilisée uniquement dans le code au sein d'un contexte d'annotations '#nullable'.



## 6. Sanity — grille uniforme (cas degenere BFS = A*)

D'abord le cas degenere : grille 11x11, cout uniforme 1, sans obstacle. Sans poids, BFS et A* trouvent le **meme** chemin (cout = nombre de pas). Verifions.

In [6]:
// Sanity : grille 11x11 uniforme (cout 1 partout). Cas degenere ou BFS = A*.
const int W = 11, H = 11;
var start = new GridState(0, 5);
var goal = new GridState(10, 5);
var noMarais = new HashSet<GridState>();
const int coutMarais = 1;  // uniforme = pas de marais effectif

// Successeurs ponderes generiques : cout = CoutCase du voisin.
Func<GridState, IEnumerable<(GridState, int)>> succ = s =>
    ACTIONS
        .Select(a => new GridState(s.X + a.dx, s.Y + a.dy))
        .Where(n => n.X >= 0 && n.X < W && n.Y >= 0 && n.Y < H)
        .Select(n => (n, CoutCase(n, noMarais, coutMarais)));

var (bfsPath, _, bfsNodes) = BFS(start, goal, succ);
var (astarPath, astarCost, astarNodes) = AStar(start, goal, succ, Manhattan);

Console.WriteLine("Sanity grille uniforme (cout 1) : BFS = A* (meme chemin). Cas degenere.");
Console.WriteLine($"  BFS : {bfsPath.Count - 1} pas, {bfsNodes} nœuds explores.");
Console.WriteLine($"  A*  : {astarPath.Count - 1} pas, cout {astarCost}, {astarNodes} nœuds explores.");
Console.WriteLine("  (sans poids, pas de discrimination entre BFS et A* — c'est normal)");


Sanity grille uniforme (cout 1) : BFS = A* (meme chemin). Cas degenere.


  BFS : 10 pas, 91 nœuds explores.


  A*  : 10 pas, cout 10, 11 nœuds explores.


  (sans poids, pas de discrimination entre BFS et A* — c'est normal)


## 7. LE point cle (#3801 Prong B) — terrain pondere

Maintenant le **terrain pondere** : un marais central (cout 10) au milieu de la grille (cout 1). Le depart et le but sont alignes **de part et d'autre du marais**. C'est ici que les algorithmes se **differencient** :
- **BFS / Greedy** minimisent le nombre de PAS -> traversent le marais tout droit (10 pas mais cout eleve).
- **A*** minimise le COUT reel -> contourne le marais (plus de pas mais cout faible).

C'est le cas **non-degenere** ou A* prouve sa valeur (sur terrain uniforme, A* = BFS).

In [7]:
// PRONG B : terrain pondere. Marais central cout 10 (sinon 1). Depart/but alignes a travers le marais.
// BFS/Greedy traversent (peu de pas, cout eleve). A* contourne (plus de pas, cout faible).
var marais = new HashSet<GridState>(
    from x in Enumerable.Range(3, 5) from y in Enumerable.Range(3, 5) select new GridState(x, y));
const int coutMaraisLourd = 10;

Func<GridState, IEnumerable<(GridState, int)>> succPondere = s =>
    ACTIONS
        .Select(a => new GridState(s.X + a.dx, s.Y + a.dy))
        .Where(n => n.X >= 0 && n.X < W && n.Y >= 0 && n.Y < H)
        .Select(n => (n, CoutCase(n, marais, coutMaraisLourd)));

var (bfsW, _, bfsWnodes) = BFS(start, goal, succPondere);
var (greedyW, _, greedyWnodes) = GreedyBestFirst(start, goal, succPondere, Manhattan);
var (astarW, astarWcost, astarWnodes) = AStar(start, goal, succPondere, Manhattan);

int CoutChemin(List<GridState> path, HashSet<GridState> mz, int cMarais) =>
    path.Sum(s => CoutCase(s, mz, cMarais));
int PasDansMarais(List<GridState> path) => path.Count(s => marais.Contains(s));

// Tableau numerique (le twin Python le fait visuellement ; ici c'est la table Pas/Cout/Nœuds).
// ATTENTION .NET : les headers statiques en PadRight (PAS d'interpolation {'text',N} = char-literal = CS1012).
Console.WriteLine("=== Terrain pondere : BFS / Greedy / A* ===");
Console.WriteLine("Algo".PadRight(18) + "Pas".PadRight(6) + "Cout".PadRight(7) + "Noeuds".PadRight(9) + "Traverse marais ?");
Console.WriteLine(new string('-', 50));
var lignes = new[] {
    ("BFS", bfsW, bfsWnodes),
    ("Greedy", greedyW, greedyWnodes),
    ("A*", astarW, astarWnodes),
};
foreach (var (nom, path, nodes) in lignes)
{
    int pas = path.Count - 1;
    int cout = CoutChemin(path, marais, coutMaraisLourd);
    int nm = PasDansMarais(path);
    string etat = nm > 0 ? $"oui ({nm} cases)" : "non";
    Console.WriteLine($"{nom,-18}{pas,-6}{cout,-7}{nodes,-9}{etat}");
}

Console.WriteLine();
Console.WriteLine($"BFS/Greedy : {bfsW.Count - 1} pas (court) mais cout {CoutChemin(bfsW, marais, coutMaraisLourd)} (traverse le marais).");
Console.WriteLine($"A*         : {astarW.Count - 1} pas (plus long) mais cout {astarWcost} (contourne le marais).");
Console.WriteLine("Seul A* minimise le COUT reel. C'est la ou A* discrimine (terrain uniforme -> A*=BFS).  ");


=== Terrain pondere : BFS / Greedy / A* ===


Algo              Pas   Cout   Noeuds   Traverse marais ?


--------------------------------------------------


BFS               10    56     91       oui (5 cases)


Greedy            10    56     11       oui (5 cases)


A*                16    17     42       non


BFS/Greedy : 10 pas (court) mais cout 56 (traverse le marais).


A*         : 16 pas (plus long) mais cout 16 (contourne le marais).


Seul A* minimise le COUT reel. C'est la ou A* discrimine (terrain uniforme -> A*=BFS).  


## 8. Admissibilite et coherence de l'heuristique

Une heuristique est **admissible** si elle ne surestime jamais le vrai cout restant. Manhattan est admissible sur grille (le chemin le plus court en pas est une minoration du cout). Une heuristique admissible **ET consistante** (h(n) <= cout(n,n') + h(n')) garantit qu'A* ne rouvre jamais un nœud ferme. Verifions la consistance de Manhattan.

In [8]:
// Admissibilite/consistance de Manhattan : pour tout pas (cout >= 1), h(n) <= cout(n,n') + h(n').
// Verif sur la grille pondere : Manhattan ne baisse jamais de plus de 1 par pas (cout min = 1).
bool consistant = true;
foreach (var s in Enumerable.Range(0, W).SelectMany(x => Enumerable.Range(0, H).Select(y => new GridState(x, y))))
{
    int hs = Manhattan(s, goal);
    foreach (var (n, cost) in succPondere(s))
    {
        int hn = Manhattan(n, goal);
        // consistance : h(s) <= cost(s->n) + h(n). cost >= 1, donc h peut baisser d'au plus cost.
        if (hs > cost + hn) { consistant = false; break; }
    }
    if (!consistant) break;
}
Console.WriteLine($"Manhattan est consistante sur la grille ponderee : {consistant}");
Console.WriteLine("(h(n) <= cout(n->n') + h(n') pour tout arc ; garantit qu'A* ne rouvre pas un nœud ferme)");


Manhattan est consistante sur la grille ponderee : True


(h(n) <= cout(n->n') + h(n') pour tout arc ; garantit qu'A* ne rouvre pas un nœud ferme)


## 9. Exercices

**Exercice 1 — Grille avec obstacles.** Ajoutez un ensemble de cases bloquees (cout infini / non traversables) et resolvez : BFS/Greedy doivent les contourner, A* idem. *Indice* : filtrez les successeurs bloques dans la fonction `succPondere`.

**Exercice 2 — Heuristique euclidienne.** Implementez une heuristique euclidienne `sqrt(dx^2+dy^2)` (arrondie). Est-elle admissible sur grille 4-connexe ? Pourquoi ? *Indice* : la distance euclidienne minore-t-elle le nombre de pas Manhattan ?

**Exercice 3 — A* avec cout variable different.** Changez le cout du marais (10 -> 5, puis 20). Observez le seuil ou A* decide de **contourner** vs **traverser**. Quel est le cout critique ? *Indice* : comparez cout(chemin droit) = 10*coutMarais vs cout(contournement) = 16.

In [9]:
// EXERCICE 1 : grille avec obstacles (cases bloquees, cout infini).
// TODO etudiant : definir un HashSet<GridState> d'obstacles et filtrer succPondere.
// Indice : dans la clause .Where, ajouter !obstacles.Contains(n).
HashSet<GridState> obstacles = new();  // TODO etudiant : remplir (ex : mur vertical)
Func<GridState, IEnumerable<(GridState, int)>> succAvecObstacles = s =>
    ACTIONS
        .Select(a => new GridState(s.X + a.dx, s.Y + a.dy))
        .Where(n => n.X >= 0 && n.X < W && n.Y >= 0 && n.Y < H)
        .Where(n => !obstacles.Contains(n))  // TODO etudiant : filtre obstacles
        .Select(n => (n, CoutCase(n, marais, coutMaraisLourd)));
Console.WriteLine("Exercice a completer : ajouter des obstacles et observer le contournement.");


Exercice a completer : ajouter des obstacles et observer le contournement.


In [10]:
// EXERCICE 2 : heuristique euclidienne (arrondie).
// TODO etudiant : implementer sqrt(dx^2+dy^2) et tester son admissibilite sur grille 4-connexe.
// Indice : euclidienne <= Manhattan (le chemin droit est plus court que les escaliers).
static int Euclidean(GridState a, GridState b) =>
    (int)Math.Round(Math.Sqrt((a.X - b.X) * (a.X - b.X) + (a.Y - b.Y) * (a.Y - b.Y)));  // TODO etudiant : verifier admissibilite
Console.WriteLine($"Euclidienne (0,0)->(3,4) = {Euclidean(new GridState(0,0), new GridState(3,4))} ; Manhattan = {Manhattan(new GridState(0,0), new GridState(3,4))}");
Console.WriteLine("Exercice a completer : euclidienne est-elle admissible sur grille 4-connexe ?");


Euclidienne (0,0)->(3,4) = 5 ; Manhattan = 7


Exercice a completer : euclidienne est-elle admissible sur grille 4-connexe ?


In [11]:
// EXERCICE 3 : seuil de cout ou A* bascule traverser/contourner.
// TODO etudiant : boucler sur coutMaraisCritique et trouver ou astarW.Count-1 passe de 10 a 16.
// Indice : cout(chemin droit) = 10*c ; cout(contournement) = 16. Seuil quand 10*c >= 16 + ...
Console.WriteLine("Exercice a completer : a partir de quel cout_marais A* decide-t-il de contourner ?");
foreach (int c in new[] { 2, 3, 4, 5, 10, 20 })
{
    // TODO etudiant : re-resoudre avec coutMaraisLourd = c et noter astarW.Count-1
    Console.WriteLine($"  cout_marais={c,2} -> (etudiant : A* fait combien de pas ?)");
}


Exercice a completer : a partir de quel cout_marais A* decide-t-il de contourner ?


  cout_marais= 2 -> (etudiant : A* fait combien de pas ?)


  cout_marais= 3 -> (etudiant : A* fait combien de pas ?)


  cout_marais= 4 -> (etudiant : A* fait combien de pas ?)


  cout_marais= 5 -> (etudiant : A* fait combien de pas ?)


  cout_marais=10 -> (etudiant : A* fait combien de pas ?)


  cout_marais=20 -> (etudiant : A* fait combien de pas ?)


## 10. Conclusion — du search nu a A*

Ce twin C# parcourt la meme progression que le twin Python :
- **BFS** : optimal en PAS (cout uniforme), explore beaucoup.
- **DFS** : ni optimal ni complet, contraste pedagogique.
- **Greedy Best-First** : guide par h seul, rapide mais non optimal.
- **A*** : `f = g + h`, optimal en COUT si h admissible (Manhattan).

**Le point cle (#3801 Prong B)** : sur terrain uniforme, BFS = A* (cas degenere). Sur **terrain pondere**, seul A* minimise le cout reel (16 pas / cout 16 en contournant le marais, vs BFS 10 pas / cout 55 en traversant). C'est la ou la capacite d'A* est visible dans la sortie — pas un exemple trivial.

> Retour au [twin Python](Planners-3-State-Space.ipynb) — qui demontre la meme chose visuellement (heatmap + chemins superposes).